# 最終課題

In [11]:
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin, urlparse
from collections import deque

## サイトマップ抽出のための設定
# 1. 武蔵野大学のトップページURLを設定
BASE_URL = 'https://www.musashino-u.ac.jp/' 
DOMAIN = urlparse(BASE_URL).netloc

# 2. 辞書型変数を初期化
sitemap = {}

# 巡回済みURLを管理 (重複アクセス防止)
visited_urls = set()

# 巡回対象URLを管理 (FIFOで管理するためdequeを使用)
urls_to_visit = deque([BASE_URL])

# 巡回する最大ページ数 (サイトが大きいため、無限ループを防ぐためのガード)
MAX_PAGES = 50 
count = 0

## サイトマップの抽出処理
print("--- サイトマップ抽出開始 ---")

while urls_to_visit and count < MAX_PAGES:
    current_url = urls_to_visit.popleft() # キューの先頭からURLを取得

    # 既に訪問済みのURLであればスキップ
    if current_url in visited_urls:
        continue
    
    # 訪問済みリストに追加
    visited_urls.add(current_url)
    count += 1
    
    print(f"\n✅ 訪問中 ({count}/{MAX_PAGES}): {current_url}")

    # 必須: Webサイトへの負荷軽減のための待機
    time.sleep(2) # 2秒待機（サーバーの負担を考慮し、必ず入れること）
    
    # ページのアクセスとエラーチェック
    try:
        response = requests.get(current_url, timeout=10)
    except requests.exceptions.RequestException as e:
        print(f"🚨 接続エラー: {e}")
        continue
        
    # HTTPステータスコードが200番台（成功）でなければスキップ
    if not (200 <= response.status_code < 300):
        print(f"⚠️ HTTPエラーコード: {response.status_code} - スキップします")
        continue
    
    # HTMLの解析
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # URLとタイトルを辞書に格納
    title_tag = soup.find('title')
    # <title>タグが取得できなかった場合は「タイトルなし」とする
    page_title = title_tag.string.strip() if title_tag and title_tag.string else "タイトルなし"
    
    # key: URL, value: <title>タグに挟まれた文字列
    sitemap[current_url] = page_title
    
    # ページ内のリンクを収集し、巡回対象に追加
    for link in soup.find_all('a', href=True):
        href = link['href']
        # 絶対URLに変換
        full_url = urljoin(current_url, href)
        parsed_url = urlparse(full_url)

        # リンクが同ドメイン内かつ、まだ巡回リストにも訪問済みリストにもない場合のみ追加
        if parsed_url.netloc == DOMAIN:
            # URLからクエリパラメータやフラグメントを削除して比較するためにクリーン化
            cleaned_url = full_url.split('#')[0].split('?')[0]
            
            if cleaned_url not in visited_urls and cleaned_url not in urls_to_visit:
                urls_to_visit.append(cleaned_url)

print("\n--- サイトマップ抽出完了 ---")
# 3. 辞書型変数を print() で表示
print("\n--- 最終的なサイトマップ ---")
print(sitemap)
print(f"\n合計で {len(sitemap)} ページを抽出しました。")

--- サイトマップ抽出開始 ---

✅ 訪問中 (1/50): https://www.musashino-u.ac.jp/

✅ 訪問中 (2/50): https://www.musashino-u.ac.jp/access.html

✅ 訪問中 (3/50): https://www.musashino-u.ac.jp/admission/request.html

✅ 訪問中 (4/50): https://www.musashino-u.ac.jp/contact.html

✅ 訪問中 (5/50): https://www.musashino-u.ac.jp/prospective-students.html

✅ 訪問中 (6/50): https://www.musashino-u.ac.jp/students.html

✅ 訪問中 (7/50): https://www.musashino-u.ac.jp/alumni.html

✅ 訪問中 (8/50): https://www.musashino-u.ac.jp/parents.html

✅ 訪問中 (9/50): https://www.musashino-u.ac.jp/business.html

✅ 訪問中 (10/50): https://www.musashino-u.ac.jp/guide/

✅ 訪問中 (11/50): https://www.musashino-u.ac.jp/guide/profile/

✅ 訪問中 (12/50): https://www.musashino-u.ac.jp/guide/activities/

✅ 訪問中 (13/50): https://www.musashino-u.ac.jp/guide/campus/

✅ 訪問中 (14/50): https://www.musashino-u.ac.jp/guide/facility/

✅ 訪問中 (15/50): https://www.musashino-u.ac.jp/guide/information/

✅ 訪問中 (16/50): https://www.musashino-u.ac.jp/guide/profile/media/

✅ 訪問中 (17/50): 

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin, urlparse

# --- 設定 ---
# 武蔵野大学のトップページURL
START_URL = "https://www.musashino-u.ac.jp/" 
# クローリング時の負荷軽減のための待機時間（秒）
WAIT_TIME = 2 
# ターゲットドメイン名
TARGET_DOMAIN = urlparse(START_URL).netloc
# --- 設定終 ---

def crawl_musashino_u_sitemap(start_url, wait_time):
    """
    武蔵野大学のWebサイトをクローリングし、サイトマップ（URLとタイトル）を生成します。
    """
    
    # 訪問済みのURLを管理するセット
    visited = set()
    # 訪問予定のURLを管理するリスト（キューとして使用）
    to_visit = [start_url]
    # 結果を格納する辞書
    sitemap = {}

    print(f"クローリング開始: {start_url}")

    while to_visit:
        # 訪問リストから次のURLを取り出す
        current_url = to_visit.pop(0)

        # 既に訪問済みであればスキップ
        if current_url in visited:
            continue

        print(f"訪問中: {current_url}")
        
        # サーバー負荷軽減のため、必ず待機する
        time.sleep(wait_time) 
        
        try:
            # ページを取得
            response = requests.get(current_url, timeout=10)
            response.raise_for_status() # HTTPエラーが発生した場合に例外を発生させる
            
            # コンテンツタイプがHTMLであることを確認（誤ってPDFなどをダウンロードしないように）
            content_type = response.headers.get('Content-Type', '')
            if 'text/html' not in content_type:
                print(f"  [SKIP] HTMLではないコンテンツ: {content_type}")
                visited.add(current_url)
                continue
                
        except requests.exceptions.RequestException as e:
            print(f"  [ERROR] アクセス失敗: {e}")
            visited.add(current_url)
            continue

        # 訪問済みリストに追加
        visited.add(current_url)
        
        # HTML解析
        soup = BeautifulSoup(response.content, 'html.parser')

        # 1. タイトルを取得し、辞書に格納
        title_tag = soup.find('title')
        page_title = title_tag.text.strip() if title_tag else "タイトルなし"
        sitemap[current_url] = page_title
        
        print(f"  [TITLE] {page_title}")

        # 2. 同一ドメインの全てのリンクを辿る
        for link in soup.find_all('a', href=True):
            href = link.get('href')
            
            # コメントアウトされていないリンクかどうかを判定する必要があるが、
            # BeautifulSoupはコメント内の要素を自動で解析しないため、基本的にはコメント外の有効なリンクのみを処理します。
            
            # 絶対URLに変換
            absolute_url = urljoin(current_url, href)
            
            # URLからフラグメント（#...）を除去
            cleaned_url = absolute_url.split('#')[0]
            
            # リンク先のドメインをチェック
            link_domain = urlparse(cleaned_url).netloc
            
            # 同一ドメインであり、かつ未訪問のURLであればリストに追加
            if link_domain == TARGET_DOMAIN and cleaned_url not in visited and cleaned_url not in to_visit:
                # 重複の追加を避けるため、to_visitとvisitedの両方になければ追加
                to_visit.append(cleaned_url)
                
    return sitemap

# 実行
musashino_u_sitemap = crawl_musashino_u_sitemap(START_URL, WAIT_TIME)

print("-" * 50)
print("--- サイトマップ抽出結果（辞書型変数） ---")
# 辞書型変数を print() で表示する
print(musashino_u_sitemap)

print("-" * 50)
print(f"合計ページ数: {len(musashino_u_sitemap)}")